In [ ]:
# 在 Google Colab 里运行时，先把本书的代码拉下来（本地 Jupyter 会跳过这一格）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/pytorch-inside-out"):
        !git clone -q --depth 1 https://github.com/wikieden/pytorch-inside-out /content/pytorch-inside-out
    %cd /content/pytorch-inside-out/code/notebooks


# 第 17 章 · 集合通信：五个动词和它们的代价

本章回答：多卡之间交换数据靠哪几种"集合操作"？all-reduce、all-gather、reduce-scatter 各是什么结果？ring all-reduce 为什么每张卡只要发约 2 倍数据？哪个并行方法用哪个操作？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/17-collectives.md`（讲解、动画、练习都在讲义里）。对应原文：Ultra-Scale Playbook §A0「Parallel Programming Crash Course」（`distributed-pretraining/materials/text/ultrascale-playbook.md`：Broadcast 第 3297 行、Reduce & AllReduce 第 3341 行、Gather & AllGather 第 3398 行、Scatter & ReduceScatter 第 3458 行、Ring AllReduce 第 3528–3575 行、Barrier 第 3576 行、NCCL 第 3608 行）；本章输出在本机 Gloo 2 进程上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import sys
sys.path.insert(0, "..")   # 子进程要能按名字导入函数，所以用本地模块

import torch
import torch.distributed as dist

import ch17_collectives as M          # 传给 run(...) 的函数从这里取
from dist_util import run
torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 all_reduce | 求和人人有 | `five_verbs` 的 all_reduce | [3, 6] |
| 第一步 all_gather | 拼接人人有 | 同上 | [[1, 2], [2, 4]] |
| 第一步 reduce_scatter | 求和各拿一段 | 同上 | rank 0 [30]，rank 1 [60] |
| 第三步 ring 通信量 | 2(N−1)K/N | `ring_allreduce_sent(1, N)` | 4 卡 1.5K，64 卡 1.969K |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `five_verbs`**（来自 `ch17_collectives.py`）：下面要用到的小工具。

In [ ]:
def five_verbs(rank, world):
    x = torch.tensor([1.0, 2.0]) * (rank + 1)
    y = x.clone()
    dist.all_reduce(y)
    g = [torch.zeros(2) for _ in range(world)]
    dist.all_gather(g, x)
    out = torch.zeros(1)
    dist.reduce_scatter_tensor(out, torch.tensor([10.0, 20.0]) * (rank + 1))
    b = torch.tensor([float(rank)])
    dist.broadcast(b, src=1)
    return dict(all_reduce=y.tolist(), all_gather=[t.tolist() for t in g], reduce_scatter=out.tolist(),
                broadcast=b.tolist())

**函数 `ring_allreduce_sent`**（来自 `ch17_collectives.py`）：ring all-reduce 每卡发送的元素数：2(N−1)K/N。

In [ ]:
def ring_allreduce_sent(K: float, N: int) -> float:
    """ring all-reduce 每卡发送的元素数：2(N−1)K/N。"""
    return 2 * (N - 1) * K / N

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 2 个进程的五个动词：")
for r, res in enumerate(run(M.five_verbs)):
    print(f"  rank {r}: {res}")

In [ ]:
print("\n第三步 每卡发送量（以 K 为单位）：4 卡", ring_allreduce_sent(1, 4), "，64 卡", round(ring_allreduce_sent(1, 64), 3))

## 3. 代码里才看得到的

1. **all_reduce 是原地的**：所以先 `clone` 一份，不然原来的 x 就没了。
2. **每个 rank 都要调用、顺序一样**：`five_verbs` 两个 rank 跑的是完全相同的代码。

## 4. 自己改一改

- `run(five_verbs, world=3)`，看三个进程的结果。
- 在 `all_reduce` 里加 `op=dist.ReduceOp.MAX`。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> all_reduce 加起来人人有，all_gather 拼起来人人有，reduce_scatter 加完各拿一段，broadcast 一份发全体；
> 大家同序调用，少一个就卡死；all_reduce 拆成两半，FSDP 在中间做事；
> ring 每卡发两倍 K，几乎不随卡数涨。